# MG628 Week 8: Inferential Statistics, Assignment Code

This notebook contains working, fully commented code for every task in the Week 8 assignment. **The code produces the numbers; interpreting them is your job.** After each task, write your interpretation in the text cell provided.

**How to use it**
1. In the first code cell, set `TRACK` to your scenario: `"Business"`, `"Healthcare"`, or `"Public Administration"`.
2. Choose **Runtime → Run all**, or run the cells from the top with **Shift + Enter**. Later cells depend on earlier ones, so run them in order.
3. Read the comments in each cell to understand what the code does and why.
4. Write your interpretation after each task.

Before you submit, choose **Runtime → Restart session and run all** to confirm every cell runs without errors.

In [ ]:
# ============================================================
# STEP 0: CHOOSE YOUR TRACK
# ============================================================
# Type your scenario exactly as written, inside the quotes:
#   "Business"               TechFlow premium onboarding
#   "Healthcare"             Telehealth recovery app
#   "Public Administration"  Fast-Track vocational training
# Every later cell reads this one setting, so this is the
# only line you need to change.
TRACK = "Business"

In [ ]:
# ============================================================
# SETUP: LIBRARIES
# ============================================================
# Libraries are toolkits other people have written. We
# "import" them once at the top so every later cell can use
# them. Colab already has all of these installed.

import pandas as pd               # tables of data (DataFrames)
import numpy as np                # math on columns of numbers
import matplotlib.pyplot as plt   # charts
from scipy import stats           # statistical tests
from scipy.stats import t as t_dist  # t distribution (for CIs)
import statsmodels.api as sm      # regression models

# Show decimals with 2 places in tables, so output is easy
# to read (this changes display only, not the data).
pd.set_option("display.float_format", "{:,.2f}".format)

In [ ]:
# ============================================================
# SETUP: TRACK SETTINGS
# ============================================================
# Each track uses a different file and different column
# names. This dictionary stores those details in one place,
# so the analysis code below works for all three tracks.
#
#   url      where the CSV lives on the MG628 GitHub
#   score    the 1-10 rating column (Tasks 3 and 6)
#   outcome  the dollar column (Tasks 4 and 5)
#   id       the ID column

RAW = ("https://raw.githubusercontent.com/"
       "monroeu/MG628/refs/heads/main/week08/")

SETTINGS = {
    "Business": {
        "url": RAW + "business_conversion_data.csv",
        "score": "Satisfaction_Score",
        "outcome": "Monthly_Spend",
        "id": "Customer_ID",
    },
    "Healthcare": {
        "url": RAW + "healthcare_recovery.csv",
        "score": "Satisfaction_Score",
        "outcome": "Recovery_Cost",
        "id": "Patient_ID",
    },
    "Public Administration": {
        "url": RAW + "employment_program.csv",
        "score": "Confidence_Score",
        "outcome": "Monthly_Income",
        "id": "Participant_ID",
    },
}

# Look up the settings for the track chosen in Step 0.
# If TRACK is misspelled, stop with a clear message instead
# of a confusing error later.
if TRACK not in SETTINGS:
    raise ValueError(
        f"TRACK must be one of {list(SETTINGS)}; "
        f"you typed {TRACK!r}"
    )
cfg = SETTINGS[TRACK]
SCORE = cfg["score"]      # e.g. "Satisfaction_Score"
OUTCOME = cfg["outcome"]  # e.g. "Monthly_Spend"
print(f"Track: {TRACK}")
print(f"Score column: {SCORE};  outcome column: {OUTCOME}")

In [ ]:
# ============================================================
# SETUP: HELPER FUNCTIONS
# ============================================================
# A function packages steps we'll reuse, so we write them
# once and call them by name.

def confidence_interval_95(data):
    """Return (mean, lower, upper) of a 95% confidence
    interval for the AVERAGE of `data`.

    Formula: mean +/- t* x SE
      SE (standard error) = SD / sqrt(n): how much the
          sample mean would vary from sample to sample
      t*  = the t value that leaves 2.5% in each tail
          (about 1.98 when n = 150)
    """
    n = len(data)                          # sample size
    mean = np.mean(data)                   # sample average
    std_err = stats.sem(data)              # SD / sqrt(n)
    t_star = t_dist.ppf(0.975, n - 1)      # critical value
    margin = std_err * t_star              # margin of error
    return mean, mean - margin, mean + margin


def money(value):
    """Format dollars with the minus sign in front:
    -356.57 -> "-$356.57" (not "$-356.57")."""
    sign = "-" if value < 0 else ""
    return f"{sign}${abs(value):,.2f}"


def cohens_d(treatment, control):
    """Effect size: the difference in means measured in
    standard deviations. Rough guide: 0.2 = small,
    0.5 = medium, 0.8 = large. Unlike the p-value, it
    does not grow just because the sample is bigger."""
    pooled_sd = np.sqrt(
        (treatment.var(ddof=1) + control.var(ddof=1)) / 2
    )
    return (treatment.mean() - control.mean()) / pooled_sd

## Task 1: Load and explore the data

In [ ]:
# ============================================================
# TASK 1: LOAD AND EXPLORE
# ============================================================
# Purpose: get the data into Python and confirm it is clean
# before any analysis. Problems missed here (missing values,
# numbers stored as text) silently break every later result.

# Read the CSV straight from GitHub into a DataFrame.
df = pd.read_csv(cfg["url"])

# First 10 rows: see what one record looks like, so you can
# describe each column in your interpretation.
display(df.head(10))

# info(): row count, column names, data types, and non-null
# counts. Score and dollar columns should be numeric
# (float64/int64). If every column shows the full row count
# as non-null, there are no missing values.
# (Called on its own line: print(df.info()) adds "None".)
df.info()

# describe(): count, mean, SD, min, quartiles, and max for
# each numeric column. Check scores stay within 1-10 and
# dollar values are plausible (no negatives).
display(df.describe())

# How many records are in each group? Confirms the split.
print("\nRecords per group:")
print(df["Group"].value_counts())

# Missing values per column (0 everywhere = complete data).
print("\nMissing values per column:")
print(df.isna().sum())

**✍️ Your interpretation (Task 1):** write your answer here, in your own words.

## Task 2: Summary statistics by group

In [ ]:
# ============================================================
# TASK 2: SUMMARY STATISTICS BY GROUP
# ============================================================
# Purpose: describe each group with its average (mean) and
# spread (standard deviation) before testing anything. These
# are DESCRIPTIVE statistics: they describe this sample only.

# groupby("Group") splits rows into Control and Treatment;
# .agg() then calculates the listed statistics per group.
summary_stats = df.groupby("Group").agg(
    {SCORE: ["mean", "std"], OUTCOME: ["mean", "std"]}
).round(2)
display(summary_stats)

# The difference between groups, always Treatment minus
# Control, so a positive number means Treatment is higher.
means = df.groupby("Group")[[SCORE, OUTCOME]].mean()
score_diff = means.loc["Treatment", SCORE] - means.loc["Control", SCORE]
outcome_diff = (
    means.loc["Treatment", OUTCOME] - means.loc["Control", OUTCOME]
)
print(f"{SCORE} difference (T - C): {score_diff:.2f} points")
print(f"{OUTCOME} difference (T - C): {money(outcome_diff)}")

# Public Administration also needs the annual difference
# and a comparison with the $3,200 self-sufficiency wage.
if TRACK == "Public Administration":
    print(f"Annual income difference: ${outcome_diff * 12:,.2f}")
    for g in ["Control", "Treatment"]:
        avg = means.loc[g, OUTCOME]
        side = "above" if avg > 3200 else "below"
        print(f"{g} average ${avg:,.2f} is {side} $3,200")

**✍️ Your interpretation (Task 2):** write your answer here, in your own words.

## Task 3: Two-sample t-test

In [ ]:
# ============================================================
# TASK 3: TWO-SAMPLE T-TEST ON THE SCORE
# ============================================================
# Purpose: decide whether the score gap between groups is
# real (would hold in the wider population) or could just be
# sampling luck.
#   H0 (null):        the two population means are equal
#   H1 (alternative): the two population means differ
# Decision rule at alpha = 0.05:
#   p < 0.05  -> reject H0 (statistically significant)
#   p >= 0.05 -> fail to reject H0 (never "accept" H0)

# Pull out each group's scores as a separate column.
control_scores = df[df["Group"] == "Control"][SCORE]
treatment_scores = df[df["Group"] == "Treatment"][SCORE]

# Independent two-sample t-test (pooled version, as in the
# assignment template). t = difference / standard error.
# NOTE: Control is entered first, so t is NEGATIVE when
# Treatment scores higher. Read direction from the means.
t_stat, p_value = stats.ttest_ind(control_scores, treatment_scores)
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4g}")   # 4g shows tiny p clearly
print("Result:", "SIGNIFICANT (p < 0.05)" if p_value < 0.05
      else "NOT SIGNIFICANT (p >= 0.05)")

# Check the equal-variance assumption behind the pooled
# test. Levene p < 0.05 suggests unequal spreads; Welch's
# test (equal_var=False) does not assume equal spreads.
levene_p = stats.levene(control_scores, treatment_scores).pvalue
t_welch, p_welch = stats.ttest_ind(
    control_scores, treatment_scores, equal_var=False
)
print(f"\nLevene's test p-value: {levene_p:.4f}")
print(f"Welch's t: {t_welch:.4f}, p = {p_welch:.4g}")

# Size of the difference: in points, and as Cohen's d.
print(f"\nMean score, Control: {control_scores.mean():.2f}")
print(f"Mean score, Treatment: {treatment_scores.mean():.2f}")
print(f"Difference (T - C): "
      f"{treatment_scores.mean() - control_scores.mean():.2f} points")
print(f"Cohen's d: {cohens_d(treatment_scores, control_scores):.2f}")

**✍️ Your interpretation (Task 3):** write your answer here, in your own words.

## Task 4: 95% confidence intervals

In [ ]:
# ============================================================
# TASK 4: 95% CONFIDENCE INTERVALS FOR THE DOLLAR OUTCOME
# ============================================================
# Purpose: estimate a range of plausible values for each
# group's TRUE average outcome, not just the sample average.
# Reading: "We are 95% confident the true average lies
# between the lower and upper bound." It describes the
# average, not individual people.

# Each group's dollar values.
control_out = df[df["Group"] == "Control"][OUTCOME]
treatment_out = df[df["Group"] == "Treatment"][OUTCOME]

# Use the helper function once per group.
mean_c, lower_c, upper_c = confidence_interval_95(control_out)
mean_t, lower_t, upper_t = confidence_interval_95(treatment_out)
print(f"Control:   lower ${lower_c:,.2f} | mean ${mean_c:,.2f} "
      f"| upper ${upper_c:,.2f}")
print(f"Treatment: lower ${lower_t:,.2f} | mean ${mean_t:,.2f} "
      f"| upper ${upper_t:,.2f}")

# Do the intervals overlap? No overlap is clear evidence of
# a real difference; overlap alone does not prove there is
# no difference.
overlap = not (upper_c < lower_t or upper_t < lower_c)
print("Intervals overlap?", overlap)

# The average difference between groups in dollars.
diff = mean_t - mean_c
print(f"\nDifference in means (T - C): {money(diff)}")

# Extra: an approximate 95% CI for the DIFFERENCE itself
# (Welch standard error). Useful for decisions: if even the
# low end clears the program cost, the case is robust.
se_diff = np.sqrt(control_out.var(ddof=1) / len(control_out)
                  + treatment_out.var(ddof=1) / len(treatment_out))
diff_low, diff_high = diff - 1.97 * se_diff, diff + 1.97 * se_diff
print(f"Approx. 95% CI for the difference: "
      f"{money(diff_low)} to {money(diff_high)}")

# Public Administration: is $3,200 inside each interval?
if TRACK == "Public Administration":
    for g, lo, hi in [("Control", lower_c, upper_c),
                      ("Treatment", lower_t, upper_t)]:
        print(f"{g} CI includes $3,200? {lo <= 3200 <= hi}")
    print(f"Annual increase: ${diff * 12:,.2f}")

**✍️ Your interpretation (Task 4):** write your answer here, in your own words.

## Task 5: Regression analysis

In [ ]:
# ============================================================
# TASK 5: MULTIPLE REGRESSION
# ============================================================
# Purpose: explain the dollar outcome using two predictors
# at once, so we can separate their effects:
#   OUTCOME = b0 + b1 x SCORE + b2 x Group_Treatment + error
# b1: change in the outcome for a 1-point higher score,
#     holding group constant
# b2: Treatment vs. Control difference for people with the
#     SAME score (the program's direct effect)

# Dummy variable: 1 for Treatment, 0 for Control. Control is
# the reference group that b2 is compared against.
df["Group_Treatment"] = (df["Group"] == "Treatment").astype(int)

# X = predictors; add_constant adds the intercept (b0).
X = sm.add_constant(df[[SCORE, "Group_Treatment"]])
y = df[OUTCOME]  # the outcome we want to explain

# Ordinary least squares: finds the b values that make the
# predictions as close as possible to the actual outcomes.
model = sm.OLS(y, X).fit()
print(model.summary())

# The numbers your interpretation needs, pulled out:
# R-squared = share of variation in the outcome explained.
print(f"\nR-squared: {model.rsquared:.4f} "
      f"({model.rsquared * 100:.1f}% of variation explained)")
for var in [SCORE, "Group_Treatment"]:
    print(f"{var}: coefficient {model.params[var]:,.2f}, "
          f"p-value {model.pvalues[var]:.4g}")

In [ ]:
# ============================================================
# TASK 5 (EXTRA): HOW THE PROGRAM CREATES VALUE
# ============================================================
# Purpose: split the total dollar difference into two paths.
#   Path 1 (through the score): b1 x score difference
#   Path 2 (direct effect):     b2
# Their sum should be close to the total difference from
# Task 2. If both coefficients are significant, the program
# works through both paths.

score_gap = treatment_scores.mean() - control_scores.mean()
path_score = model.params[SCORE] * score_gap
path_direct = model.params["Group_Treatment"]
print(f"Path through the score: {model.params[SCORE]:,.2f} "
      f"x {score_gap:.2f} = {money(path_score)}")
print(f"Direct (group) effect:  {money(path_direct)}")
print(f"Sum: {money(path_score + path_direct)}  vs. "
      f"total difference {money(diff)}")

**✍️ Your interpretation (Task 5):** write your answer here, in your own words.

## Task 6: Visualizations

In [ ]:
# ============================================================
# TASK 6: HISTOGRAM AND BOX PLOT OF THE SCORE
# ============================================================
# Purpose: SEE the difference the t-test measured.
#   Histogram: where each group's scores pile up, and how
#              the two distributions overlap.
#   Box plot:  box = middle 50% (Q1 to Q3), line = median,
#              whiskers = furthest points within 1.5 x IQR,
#              dots = outliers beyond the whiskers.

# Two charts stacked vertically (easier to read on phones).
fig, ax = plt.subplots(2, 1, figsize=(7, 9))

# Same bins for both groups (10 equal bins from 1 to 10) so
# the bars line up; alpha=0.6 makes the overlap visible.
bins = np.linspace(1, 10, 11)
ax[0].hist(control_scores, bins=bins, alpha=0.6,
           label="Control", color="#3D5A80", edgecolor="white")
ax[0].hist(treatment_scores, bins=bins, alpha=0.6,
           label="Treatment", color="#E9A23B", edgecolor="white")
ax[0].set_xlabel(SCORE.replace("_", " "))
ax[0].set_ylabel("Frequency")
ax[0].set_title(f"Distribution of {SCORE.replace('_', ' ')}")
ax[0].legend()

# Box plot by group. Passing ax= keeps it in our figure;
# without it, pandas opens a new figure and leaves the
# second chart empty (a bug in the assignment template).
df.boxplot(column=SCORE, by="Group", ax=ax[1])
ax[1].set_xlabel("Group")
ax[1].set_ylabel(SCORE.replace("_", " "))
ax[1].set_title(f"{SCORE.replace('_', ' ')} by Group")
plt.suptitle("")      # remove pandas' automatic extra title
plt.tight_layout()    # stop labels overlapping
plt.show()

In [ ]:
# ============================================================
# TASK 6 (EXTRA): NUMBERS BEHIND THE CHARTS
# ============================================================
# Purpose: back up what you see with exact values: medians,
# spread (IQR), share of high and low scores, and outliers
# (the dots on the box plot).

for g, sc in [("Control", control_scores),
              ("Treatment", treatment_scores)]:
    q1, q3 = sc.quantile([0.25, 0.75])   # quartiles
    iqr = q3 - q1                         # middle-50% width
    low_fence, high_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = sc[(sc < low_fence) | (sc > high_fence)]
    print(f"{g}: median {sc.median():.2f}, IQR {iqr:.2f}, "
          f"scored 8+: {(sc >= 8).mean():.1%}, "
          f"scored 5 or less: {(sc <= 5).mean():.1%}")
    print(f"   outliers: {sorted(outliers.round(2).tolist())}")

**✍️ Your interpretation (Task 6):** write your answer here, in your own words.

## Task 7: Cost-benefit numbers for your memo

The code below calculates the numbers your memo needs. The memo itself (the decision, reasoning, recommendation, and limitations) is yours to write in a text cell.

In [ ]:
# ============================================================
# TASK 7: COST-BENEFIT CALCULATIONS
# ============================================================
# Purpose: turn the statistical results into the economic
# numbers a decision-maker needs. Each track has its own
# cost structure, so only your track's block runs.

if TRACK == "Business":
    # Program cost: $50 per customer, paid once.
    cost = 50
    gain = diff   # extra monthly spend per customer (Task 4)
    print(f"Extra monthly spend per customer: ${gain:,.2f}")
    print(f"Low end of the 95% CI for the gain: ${diff_low:,.2f}")
    # Payback: months of extra spending needed to repay $50.
    print(f"Months to recover $50: {cost / gain:.2f} "
          f"(at the low end: {cost / diff_low:.2f})")
    # Example scale-up: change new_customers to test others.
    new_customers = 1000
    first_year = new_customers * (12 * gain - cost)
    print(f"First-year net gain for {new_customers:,} customers "
          f"(if the gain lasts 12 months): ${first_year:,.0f}")

elif TRACK == "Healthcare":
    # Savings per patient: Control cost minus Treatment cost
    # (positive = telehealth is cheaper).
    saving = -diff
    print(f"Savings per patient: ${saving:,.2f}")
    # YOUR ASSUMPTIONS: replace these with your own estimates
    # and explain them in your memo.
    licensing = 0      # app licensing per year ($)
    training = 0       # staff training, one-time ($)
    it_support = 0     # IT support per year ($)
    patients_per_year = 0   # eligible surgical patients
    implementation = licensing + training + it_support
    if implementation > 0 and patients_per_year > 0:
        annual = saving * patients_per_year
        print(f"Implementation cost: ${implementation:,.0f}")
        print(f"Annual savings: ${annual:,.0f}")
        print(f"Net benefit: ${annual - implementation:,.0f}")
        print(f"Break-even: {implementation / saving:,.0f} patients")
    else:
        print("Enter your cost and patient assumptions above, "
              "then re-run this cell.")

else:  # Public Administration
    # Program cost: $5,000 per participant.
    cost = 5000
    gain_month = diff
    gain_year = 12 * gain_month
    print(f"Monthly income increase: ${gain_month:,.2f}")
    print(f"Annual income increase: ${gain_year:,.2f}")
    # Payback: months of extra income to equal $5,000.
    print(f"Payback: {cost / gain_month:.1f} months")
    # ROI: annual gain as a percentage of the cost.
    print(f"Annual ROI: {gain_year / cost * 100:.1f}%")
    # Share earning above the $3,200 self-sufficiency wage,
    # counted directly and estimated with a normal curve.
    for g, inc in [("Control", control_out),
                   ("Treatment", treatment_out)]:
        counted = (inc > 3200).mean()
        normal_est = 1 - stats.norm.cdf(3200, inc.mean(), inc.std())
        print(f"{g}: {counted:.1%} above $3,200 (counted), "
              f"{normal_est:.1%} (normal estimate)")
    extra = ((treatment_out > 3200).mean()
             - (control_out > 3200).mean()) * 150
    print(f"Additional participants above $3,200 per 150: {extra:.0f}")

**✍️ Your memo (Task 7):** write your memo here in Markdown: To / From / Date / Subject, your YES or NO decision, key findings with numbers, cost-benefit analysis, recommendation, and limitations.

Tip: in text cells, type `\$` for a dollar sign. Colab treats text between two plain `$` signs as a math formula.